# 1. Grad

**Refers to if the tensors have the `requires_grad` flag, and can be:**
1. True (The Gradient are present in the tensor)
2. False (The Gradient aren't present in the tensor)

### 0. Creating a tensor with `torch.tensor(..., requires_grad=True)`
### 1. Creating a tensor with `torch.tensor(.., requires_grad=False)` NOTE THIS WAS THE DEFAULT OPTION
### 2. Converting a tensor's `tensorX = tensorX.requires_grad(True/False)`
- Don't have inplace.

When `requires_grad` it's `True` that means it's active in the tensor, PyTorch spends more memory and processing. Why does it get more weight?When you enable requirements_grad=True, PyTorch tracks all operations that are with that tensor. This creates a computer graph (computer graph).

> We saw in /Defaults how to put one of them as a default for not do this everytime. Use `torch.set_default_tensor_type()` or set it globally when creating tensors in your training loop.

In [1]:
import torch

x1 = torch.randn(3, 3)
print(x1.requires_grad) 

False


In [2]:
import torch

x2 = torch.randn(3, 3, requires_grad=True)
print(x2.requires_grad) 

True


In [3]:
x1.requires_grad_(True)
print(x1.requires_grad)

True


### See the differences

In [4]:
import time

start1 = time.time()

tensor1 = torch.randn((10000,10000), requires_grad=True)
tensor2 = torch.randn((10000,10000), requires_grad=True)

torch3 = torch.matmul(tensor1, tensor2)

end1 = time.time()


print(f"Time: {end1 - start1:.4f} seconds")

Time: 63.8891 seconds


In [5]:
import time

start2 = time.time()

tensor1 = torch.randn((10000,10000),requires_grad=False)
tensor2 = torch.randn((10000,10000),requires_grad=False)

torch3 = torch.matmul(tensor1, tensor2)

end2 = time.time()

print(f"Time: {end2 - start2:.4f} seconds")

Time: 56.4612 seconds


In [7]:
print(f"The difference between CPU vs GPU it's: {(end1 - start1) - (end2 - start2):.1f} seconds")

The difference between CPU vs GPU it's: 7.4 seconds


# 2. Why this happens?

When you inicialize a tensor with the grad `True`, the torch will separates a place in the memory to allocate the gradient numbers. In the start the gradient it's 0 and goes accumulation

We can starting the accumulation with the `.backward()`, thats it's a function in PyTorch wheres goes do the derivate and calculate the gradient for each parameter

> If use the .backward() will calculate the gradient for each parameters. And now the number of the gradient goes allocate in the memory. If I use that parameter in other function and calculate the gradient again, will sum with the gradient allocate in the memory

In [26]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

y.backward() # will calculate the derivates for all the parameters in the function y

After do the derivates and calculate the gradient for all the parameters we can acess them with the `.grad()`

In [ ]:
print(f'Gradient of X: {x.grad}')
# dY/DX = 2x
# x = 2 -> 4

Gradient of X: 4.0


See the accumulation

In [ ]:
z = 2 * x ** 2
z.backward()

print(f'Gradient of X accumulation: {x.grad}')
# dZ/dX = 4x
# x = 2 -> 8
# 4 + 8 = 12

Gradient of X accumulation: 12.0


For some situations it's good the reset the gradient, and we can use `.grad.zero_()` for a specific parameter

In [29]:
x.grad.zero_()
print(f'Gradient of X after the reset: {x.grad}')

Gradient of X after the reset: 0.0


We also have the `zero.grad()` for the optimizer that we'll see soon

# 3. Disable the Gradient

In the machine learning world, we use the gradient for the optimization, so we need to allow them. But when the model it's ready, the gradient will just allocate unnecessary memory and make optimization difficult. So instead of we create a lot of things, we can just disable them with the `.no_grad()`  

> The gradient are just 'disable' in the loop with no grad!

For see if the graph are created we can see at: `grad_fn`

In [43]:
x = torch.tensor(10.0, requires_grad=True)

# With grad 
f = x ** 2
print(f"grad_fn: {f.grad_fn}")          
print(f"requires_grad: {f.requires_grad}") 

# Without grad 
with torch.no_grad():
    f_no_grad = x ** 2
    print(f"grad_fn: {f_no_grad.grad_fn}")           
    print(f"requires_grad: {f_no_grad.requires_grad}")  

grad_fn: <PowBackward0 object at 0x0000023C61F38C70>
requires_grad: True
grad_fn: None
requires_grad: False
